In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
p = pathlib.Path.cwd()
for q in (p, *p.parents):
    s = q / "src" / "ftbp"   # <- change "ftbp" if you rename the package
    if s.exists():
        sys.path.insert(0, str(s.parent))  # add .../src
        break
else:
    raise RuntimeError("src/ftbp not found")

In [ ]:
import numpy as np
import pandas as pd
import itertools
from math import comb
from scipy.optimize import brentq
from scipy.stats import norm, cauchy, uniform
from scipy.integrate import quad
from ftbp.score import *

In [ ]:
### the gap experiment
from tqdm import tqdm
ns = [500, 1000, 2000]
# ns = [100]
M = 100
seeds = [53 + i for i in range(M)]
dists = [norm, cauchy]
alternatives = [-2, -1.5, -1, -.5, -.25, -.1, 0, .1, .25, .5, 1, 1.5, 2]
loss_type = 'huber'  # or 'logcosh'
if loss_type == 'huber':
    delta = 1.345
elif loss_type == 'logcosh':
    delta = 1.2047
elif loss_type == 'concordant':
    delta = 1.479
null = False


all_results = []
for alternative in alternatives:
    for n in ns:
        for dist in dists:
            for seed in tqdm(seeds):
                np.random.seed(seed)
                phi0 = 0
                while phi0 == 0:
                    # sign = np.random.choice([-1, 1])
                    x = dist.rvs(size=n) + alternative
                    phi0 = wald_test(x, null=null, delta=delta, loss_type=loss_type)
                    if phi0 == 1:
                        grid = np.linspace(np.min(x) - 10, np.max(x) + 10, 1000)
                        bp_ub_power = bound_power_upper(x, null=null, delta=delta, loss_type=loss_type)
                        bp_lb_power = bound_power_lower(x, null=null, delta=delta, loss_type=loss_type)
                        # print("UB:", bp_ub_power, "LB:", bp_lb_power)

                all_results.append({
                    'n': n,
                    'dist': getattr(dist, 'name', dist.__class__.__name__),
                    'seed': seed,
                    'power_lb': bp_lb_power,
                    'power_ub': bp_ub_power,
                    'alternative': alternative
                })

df = pd.DataFrame(all_results)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# assume df_avg already exists from previous steps:
#   columns=['n','Effect Size','Lower Bound of Power','Upper Bound of Power']
sns.set_context('talk')
sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})  # e.g. “darkgrid”, “ticks”, “white”
palette = sns.color_palette("husl", 3)

df_normal = df[df['dist'] == 'norm']

# 1. Melt to long form
df_avg = (
    df_normal
    .groupby(['n', 'alternative'], as_index=False)
    .agg({'power_ub': 'mean', 'power_lb': 'mean'})
)

# 3. Rename for paper‐friendly labels
df_avg = df_avg.rename(columns={
    'alternative': 'Effect Size',
    'power_ub':    'Upper Bound of BP',
    'power_lb':    'Lower Bound of BP'
})

marker_map = {500: 'o', 1000: 's', 2000: '^'}

# 3. Single‐plot
plt.figure(figsize=(10, 6))
ax = sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Lower Bound of BP',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend='full',
    alpha=0.8,
)

sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Upper Bound of BP',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend=False,
    alpha=0.8,
)

i = -1
for n_val in [500, 1000, 2000]:
    i += 1
    sub = df_avg[df_avg['n'] == n_val]
    ax.fill_between(sub['Effect Size'],
                    sub['Lower Bound of BP'],
                    sub['Upper Bound of BP'],
                    alpha=0.1, color=palette[i])

plt.xlabel(r'Effect Size $\theta$')
plt.ylabel(r'$m$')
ax.legend(loc='upper center', bbox_to_anchor=(0.478, -0.15), ncols=7, frameon=False)
plt.tight_layout()
# plt.show()
plt.savefig('bp_reject_m_score.pdf', bbox_inches='tight')

In [ ]:
df_avg

In [ ]:
marker_map = {500: 'o', 1000: 's', 2000: '^'}

# make y-axis to bound/n, fraction
df_avg['Lower Bound of BP ratio'] = df_avg['Lower Bound of BP'] / df_avg['n']
df_avg['Upper Bound of BP ratio'] = df_avg['Upper Bound of BP'] / df_avg['n']

# 3. Single‐plot
plt.figure(figsize=(10, 6))
ax = sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Lower Bound of BP ratio',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend='full',
    alpha=0.8,
)

sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Upper Bound of BP ratio',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend=False,
    alpha=0.8,
)

i = -1
for n_val in [500, 1000, 2000]:
    i += 1
    sub = df_avg[df_avg['n'] == n_val]
    ax.fill_between(sub['Effect Size'],
                    sub['Lower Bound of BP ratio'],
                    sub['Upper Bound of BP ratio'],
                    alpha=0.1, color=palette[i])

plt.xlabel(r'Effect Size $\theta$')
plt.ylabel('Breakdown Point of Rejection')
ax.legend(loc='upper center', bbox_to_anchor=(0.478, -0.15), ncols=7, frameon=False)
plt.tight_layout()
# plt.show()
plt.savefig(f'bp_reject_m_score_ratio.pdf', bbox_inches='tight')

In [ ]:
### the gap experiment
from tqdm import tqdm
ns = [500, 1000, 2000]
# ns = [100]
M = 100
seeds = [53 + i for i in range(M)]
dists = [norm]
alternatives = [-2, -1.5, -1, -.5, -.25, -.1, 0, .1, .25, .5, 1, 1.5, 2]
loss_type = 'huber'  # or 'logcosh'
if loss_type == 'huber':
    delta = 1.345
elif loss_type == 'logcosh':
    delta = 1.2047
elif loss_type == 'concordant':
    delta = 1.479
null = True


all_results = []
for alternative in alternatives:
    for n in ns:
        for dist in dists:
            for seed in tqdm(seeds):
                np.random.seed(seed)
                phi0 = 0
                while phi0 == 0:
                    # sign = np.random.choice([-1, 1])
                    x = dist.rvs(size=n) + alternative
                    phi0 = wald_test(x, null=null, delta=delta, loss_type=loss_type)
                    if phi0 == 1:
                        grid = np.linspace(np.min(x) - 10, np.max(x) + 10, 1000)
                        bp_ub_power = bound_power_upper(x, null=null, delta=delta, loss_type=loss_type)
                        bp_lb_power = bound_power_lower(x, null=null, delta=delta, loss_type=loss_type)
                        # print("UB:", bp_ub_power, "LB:", bp_lb_power)

                all_results.append({
                    'n': n,
                    'dist': getattr(dist, 'name', dist.__class__.__name__),
                    'seed': seed,
                    'power_lb': bp_lb_power,
                    'power_ub': bp_ub_power,
                    'alternative': alternative
                })

df = pd.DataFrame(all_results)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# assume df_avg already exists from previous steps:
#   columns=['n','Effect Size','Lower Bound of Power','Upper Bound of Power']
sns.set_context('talk')
sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})  # e.g. “darkgrid”, “ticks”, “white”
palette = sns.color_palette("husl", 3)

df_normal = df[df['dist'] == 'norm']

# 1. Melt to long form
df_avg = (
    df_normal
    .groupby(['n', 'alternative'], as_index=False)
    .agg({'power_ub': 'mean', 'power_lb': 'mean'})
)

# 3. Rename for paper‐friendly labels
df_avg = df_avg.rename(columns={
    'alternative': 'Effect Size',
    'power_ub':    'Upper Bound of BP',
    'power_lb':    'Lower Bound of BP'
})

marker_map = {500: 'o', 1000: 's', 2000: '^'}

# 3. Single‐plot
plt.figure(figsize=(10, 6))
ax = sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Lower Bound of BP',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend='full',
    alpha=0.8,
)

sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Upper Bound of BP',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend=False,
    alpha=0.8,
)

i = -1
for n_val in [500, 1000, 2000]:
    i += 1
    sub = df_avg[df_avg['n'] == n_val]
    ax.fill_between(sub['Effect Size'],
                    sub['Lower Bound of BP'],
                    sub['Upper Bound of BP'],
                    alpha=0.1, color=palette[i])

plt.xlabel(r'Effect Size $\theta$')
plt.ylabel(r'$m$')
ax.legend(loc='upper center', bbox_to_anchor=(0.478, -0.15), ncols=7, frameon=False)
plt.tight_layout()
# plt.show()
plt.savefig('bp_reject_m_score_restricted.pdf', bbox_inches='tight')

In [ ]:
df_avg

In [ ]:
marker_map = {500: 'o', 1000: 's', 2000: '^'}

# make y-axis to bound/n, fraction
df_avg['Lower Bound of BP ratio'] = df_avg['Lower Bound of BP'] / df_avg['n']
df_avg['Upper Bound of BP ratio'] = df_avg['Upper Bound of BP'] / df_avg['n']

# 3. Single‐plot
plt.figure(figsize=(10, 6))
ax = sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Lower Bound of BP ratio',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend='full',
    alpha=0.8,
)

sns.lineplot(
    data=df_avg,
    x='Effect Size', y='Upper Bound of BP ratio',
    hue='n',                 # color by bound type
    style='n',            # line style by n
    markers=marker_map,
    palette=palette,
    legend=False,
    alpha=0.8,
)

i = -1
for n_val in [500, 1000, 2000]:
    i += 1
    sub = df_avg[df_avg['n'] == n_val]
    ax.fill_between(sub['Effect Size'],
                    sub['Lower Bound of BP ratio'],
                    sub['Upper Bound of BP ratio'],
                    alpha=0.1, color=palette[i])

plt.xlabel(r'Effect Size $\theta$')
plt.ylabel('Breakdown Point of Rejection')
ax.legend(loc='upper center', bbox_to_anchor=(0.478, -0.15), ncols=7, frameon=False)
plt.tight_layout()
# plt.show()
plt.savefig(f'bp_reject_m_score_restricted_ratio.pdf', bbox_inches='tight')